In [2]:
import os
import json
import chromadb

class FAQAgent:
    def __init__(self):
        # Initializes a persistent vector database locally
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = self.chroma_client.get_or_create_collection(name="kb_collection")
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        # If documents are already loaded, don't attempt to load anything new
        if self.collection.count() > 0:
            print("📦 Local Knowledge Base loaded successfully from disk.")
            return

        print("🔄 Initializing local dataset records...")
        
        # 🟢 LOCAL FALLBACK DATASET (Bypasses Hugging Face completely)
        fallback_documents = [
            "Paris is the capital and most populous city of France, famous for its culture and history.",
            "New Delhi is the capital of India, acting as the structural hub for its governing bodies.",
            "Photosynthesis is a chemical process used by plants to convert light energy into chemical energy.",
            "To reset your system user password, navigate to account parameters, trigger an email link, and clear the session cache.",
            "Standard billing parameters dictate that invoices are generated on the 1st of every calendar month.",
            "Debit cards can be unlocked via the customer security profile or by calling the instant desk verification hotline."
        ]
        fallback_ids = [f"kb_fallback_{i}" for i in range(len(fallback_documents))]

        # Check if you have the local SQUAD dataset available
        squad_path = "../datasets/Squad_Dataset/train-v2.0.json"
        
        if os.path.exists(squad_path):
            try:
                with open(squad_path, "r", encoding="utf-8") as f:
                    squad_data = json.load(f)
                
                documents, ids = [], []
                doc_count = 0
                
                for topic in squad_data.get("data", []):
                    for paragraphs in topic.get("paragraphs", []):
                        context = paragraphs.get("context", "")
                        if context:
                            documents.append(context)
                            ids.append(f"sq_{doc_count}")
                            doc_count += 1
                        if doc_count >= 150: break
                    if doc_count >= 150: break

                if documents:
                    self.collection.add(documents=documents, ids=ids)
                    print(f"✅ Successfully loaded {len(documents)} context blocks from local SQUAD dataset.")
                    return
            except Exception as e:
                print(f"⚠️ Failed reading SQUAD JSON ({e}). Falling back to static data entries.")

        # If no local JSON dataset exists, push the hardcoded static entries instantly
        self.collection.add(documents=fallback_documents, ids=fallback_ids)
        print("✅ Successfully loaded structural fallback data entries into ChromaDB.")

    def retrieve_context(self, query: str):
        try:
            results = self.collection.query(query_texts=[query], n_results=1)
            if results and results.get('documents') and len(results['documents'][0]) > 0:
                return results['documents'][0][0]
        except Exception:
            pass
        return "No explicit matching company structural background documentation located."